# Playwright 浏览器自动化

Playwright 是现代端到端测试和浏览器自动化工具，支持 Chromium、Firefox 和 WebKit。这里优先讲定位器、自动等待、断言、网络拦截和截图这些更贴近现代 Web 测试的能力。


## 安装和浏览器

安装 Python 包后，还需要下载浏览器二进制。命令行示例：

```zsh
uv add playwright
uv run playwright install
```

如果写端到端测试，官方推荐配合 `pytest-playwright`；如果只是脚本化浏览器自动化，直接使用 `playwright.sync_api` 或 `playwright.async_api` 即可。


## 同步 API：打开页面并断言

Playwright 的定位器会自动等待元素变为可操作状态，通常不需要手写 `sleep()`。


In [ ]:
import re

from playwright.sync_api import expect, sync_playwright


with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto("https://playwright.dev/python/")

    expect(page).to_have_title(re.compile("Playwright"))
    expect(page.get_by_role("link", name="Docs")).to_be_visible()

    browser.close()


## 定位器优先级

现代测试优先使用用户可感知的定位方式，例如 role、label、text 和 test id。只有在页面没有语义信息时，才退回 CSS 或 XPath。


In [ ]:
from playwright.sync_api import expect, sync_playwright


with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto("https://demo.playwright.dev/todomvc")

    page.get_by_placeholder("What needs to be done?").fill("Learn Playwright")
    page.keyboard.press("Enter")

    expect(page.get_by_text("Learn Playwright")).to_be_visible()
    browser.close()


## 异步 API

如果项目本身使用 `asyncio`，优先使用异步 API，避免在线程池或事件循环之间来回切换。


In [ ]:
import asyncio

from playwright.async_api import async_playwright, expect


async def main() -> None:
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto("https://playwright.dev/python/")
        await expect(page.get_by_role("heading", name="Playwright for Python")).to_be_visible()
        await browser.close()


asyncio.run(main())


## 网络拦截

Playwright 可以拦截请求，常用于替换第三方接口、屏蔽慢资源或让测试数据稳定。


In [ ]:
from playwright.sync_api import Route, sync_playwright


def block_images(route: Route) -> None:
    if route.request.resource_type == "image":
        route.abort()
    else:
        route.continue_()


with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    page.route("**/*", block_images)
    page.goto("https://playwright.dev/python/")
    print(page.title())
    browser.close()


## 截图和调试

截图、trace viewer 和 codegen 能显著减少调试成本。脚本中可以用 `page.screenshot()` 保存状态；测试中可以开启 trace，用 `playwright show-trace` 回放。


In [ ]:
from pathlib import Path

from playwright.sync_api import sync_playwright


output = Path("playwright-home.png")
with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(viewport={"width": 1280, "height": 720})
    page.goto("https://playwright.dev/python/")
    page.screenshot(path=output, full_page=True)
    browser.close()

print(output)
